# Milestone 12: Architectural Efficiency & Central Focus — EfficientNet-B0
**ES 335: Machine Learning Course Project — IIT Gandhinagar**  
**Authors:** Devarsh Patel (24110245) & Dhrumil Bhutaiya (24110075)

---
### Dual Scientific Investigation:
1. **Parameter Efficiency:** Test **EfficientNet-B0 (~5.3M parameters)** to mitigate overparameterization relative to ResNet-50 (25.6M).
2. **Track B Central Diagnostic Focus (48×48 Crop):** Test whether restricting the receptive field to the central tumor ground-truth region (with 2× optical magnification) improves specificity over full 96×96 spatial context.

**Standardized Evaluation Protocol (Identical across all project models):**
- Training Split: **10,000 patches** (50% tumour / 50% normal, `seed=42`)
- Validation Split: **2,500 patches** (`seed=42`)
- Sealed Test Split: **5,000 patches** (2,500 normal / 2,500 tumour, evaluated strictly once)

In [1]:
import subprocess
result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                       capture_output=True, text=True)
print('GPU Info:', result.stdout.strip())

import torch
print(f'PyTorch Version : {torch.__version__}')
print(f'CUDA Available  : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f'  GPU {i}: {props.name} | VRAM: {props.total_memory / 1e9:.1f} GB')

GPU Info: Tesla T4, 15360 MiB
PyTorch Version : 2.1.2+cu121
CUDA Available  : True
  GPU 0: Tesla T4 | VRAM: 15.8 GB


In [2]:
import os
import random
import time
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models

from sklearn.metrics import (
    accuracy_score, roc_auc_score, confusion_matrix,
    precision_recall_curve, auc, brier_score_loss,
    ConfusionMatrixDisplay, roc_curve
)

OUTPUT_DIR  = '/kaggle/working/outputs/efficientnet'
RESULTS_DIR = '/kaggle/working/results'
PLOTS_DIR   = '/kaggle/working/plots'

os.makedirs(OUTPUT_DIR,  exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR,   exist_ok=True)

N_TRAIN     = 10000
N_VAL       = 2500
N_TEST      = 5000
EPOCHS      = 15
BATCH_SIZE  = 128
LR          = 3e-4
WEIGHT_DECAY= 1e-4
SEED        = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {DEVICE} ({torch.cuda.device_count()} GPU(s))')
print(f'Budget : Train={N_TRAIN:,} | Val={N_VAL:,} | Test={N_TEST:,} | Seed={SEED}')

Device : cuda (1 GPU(s))
Budget : Train=10,000 | Val=2,500 | Test=5,000 | Seed=42


In [3]:
BASE_DIR = '/kaggle/input/datasets/andrewmvd/metastatic-tissue-classification-patchcamelyon'
if not os.path.exists(BASE_DIR):
    BASE_DIR = '/kaggle/input/metastatic-tissue-classification-patchcamelyon'

SPLIT_FILES = {
    'train': {
        'x': f'{BASE_DIR}/pcam/training_split.h5',
        'y': f'{BASE_DIR}/Labels/Labels/camelyonpatch_level_2_split_train_y.h5'
    },
    'valid': {
        'x': f'{BASE_DIR}/pcam/validation_split.h5',
        'y': f'{BASE_DIR}/Labels/Labels/camelyonpatch_level_2_split_valid_y.h5'
    },
    'test': {
        'x': f'{BASE_DIR}/pcam/test_split.h5',
        'y': f'{BASE_DIR}/Labels/Labels/camelyonpatch_level_2_split_test_y.h5'
    }
}

def load_pcam_h5(split, n_samples, seed=42):
    x_file = SPLIT_FILES[split]['x']
    y_file = SPLIT_FILES[split]['y']
    
    print(f'Loading {split} split...')
    with h5py.File(x_file, 'r') as hx, h5py.File(y_file, 'r') as hy:
        x_key = list(hx.keys())[0]
        y_key = list(hy.keys())[0]
        
        total = hx[x_key].shape[0]
        rng = np.random.default_rng(seed)
        idx = np.sort(rng.choice(total, size=min(n_samples, total), replace=False))
        
        X = hx[x_key][idx]
        y = hy[y_key][idx].flatten().astype(np.int64)
        
    print(f'  [OK] {split:5s}: X={X.shape}, y={y.shape} (Normal={np.sum(y==0):,}, Tumour={np.sum(y==1):,})')
    return X, y

print('=' * 65)
print('Loading PCam splits (exact same 10k/2.5k/5k samples & seed=42)...')
print('=' * 65)

X_train, y_train = load_pcam_h5('train', N_TRAIN, seed=SEED)
X_val,   y_val   = load_pcam_h5('valid', N_VAL,   seed=SEED)
X_test,  y_test  = load_pcam_h5('test',  N_TEST,  seed=SEED)

print('=' * 65)
print('All splits loaded successfully.')

Loading PCam splits (exact same 10k/2.5k/5k samples & seed=42)...
Loading train split...
  [OK] train: X=(10000, 96, 96, 3), y=(10000,) (Normal=5,000, Tumour=5,000)
Loading valid split...
  [OK] valid: X=(2500, 96, 96, 3), y=(2500,) (Normal=1,250, Tumour=1,250)
Loading test split...
  [OK] test:  X=(5000, 96, 96, 3), y=(5000,) (Normal=2,500, Tumour=2,500)
All splits loaded successfully.


In [4]:
def extract_central_focus(img_rgb, crop_size=48, target_size=96):
    h, w = img_rgb.shape[:2]
    start_y = (h - crop_size) // 2
    start_x = (w - crop_size) // 2
    cropped = img_rgb[start_y:start_y + crop_size, start_x:start_x + crop_size]
    pil_img = Image.fromarray(cropped, mode='RGB')
    return pil_img.resize((target_size, target_size), Image.Resampling.BILINEAR)

class PCamCentralFocusDataset(Dataset):
    def __init__(self, X, y, crop_size=48, target_size=96, transform=None):
        self.X = X
        self.y = y.astype(np.float32)
        self.crop_size = crop_size
        self.target_size = target_size
        self.transform = transform

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        pil_img = extract_central_focus(self.X[idx], self.crop_size, self.target_size)
        if self.transform:
            tensor_img = self.transform(pil_img)
        else:
            tensor_img = transforms.ToTensor()(pil_img)
        return tensor_img, self.y[idx]

def build_transforms():
    normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                     std=[0.229, 0.224, 0.225])
    train_tf = transforms.Compose([
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomVerticalFlip(p=0.5),
        transforms.RandomChoice([
            transforms.RandomRotation((0,   0)),
            transforms.RandomRotation((90,  90)),
            transforms.RandomRotation((180, 180)),
            transforms.RandomRotation((270, 270)),
        ]),
        transforms.ColorJitter(brightness=0.1, contrast=0.1),
        transforms.ToTensor(),
        normalize,
    ])
    eval_tf = transforms.Compose([
        transforms.ToTensor(),
        normalize,
    ])
    return train_tf, eval_tf

train_tf, eval_tf = build_transforms()
train_ds = PCamCentralFocusDataset(X_train, y_train, crop_size=48, target_size=96, transform=train_tf)
val_ds   = PCamCentralFocusDataset(X_val,   y_val,   crop_size=48, target_size=96, transform=eval_tf)
test_ds  = PCamCentralFocusDataset(X_test,  y_test,  crop_size=48, target_size=96, transform=eval_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4, pin_memory=True)

print(f'Train Loader : {len(train_ds):,} samples ({len(train_loader)} batches)')
print(f'Val Loader   : {len(val_ds):,} samples')
print(f'Test Loader  : {len(test_ds):,} samples (sealed)')

Train Loader : 10,000 samples (79 batches)
Val Loader   : 2,500 samples
Test Loader  : 5,000 samples (sealed)


In [5]:
def build_efficientnet_b0():
    model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
    in_features = model.classifier[1].in_features
    model.classifier = nn.Sequential(
        nn.Dropout(p=0.3, inplace=True),
        nn.Linear(in_features, 1)
    )
    return model

model = build_efficientnet_b0().to(DEVICE)
if torch.cuda.device_count() > 1:
    model = nn.DataParallel(model)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print('=' * 60)
print(f'Model          : EfficientNet-B0 (Pretrained ImageNet Backbone)')
print(f'Total Params   : {total_params / 1e6:.2f}M  (vs ResNet-50: 25.6M | ResNet-18: 11.7M)')
print(f'Trainable      : {trainable_params / 1e6:.2f}M')
print(f'Classifier Head: Dropout(0.3) -> Linear(1280 -> 1)')
print('=' * 60)

Model          : EfficientNet-B0 (Pretrained ImageNet Backbone)
Total Params   : 4.01M  (vs ResNet-50: 25.6M | ResNet-18: 11.7M)
Trainable      : 4.01M
Classifier Head: Dropout(0.3) -> Linear(1280 -> 1)


In [6]:
def train_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    total_loss, preds, targets = 0.0, [], []
    for X, y in loader:
        X = X.to(device, non_blocking=True)
        y = y.float().unsqueeze(1).to(device, non_blocking=True)
        optimizer.zero_grad()
        if scaler:
            with torch.amp.autocast('cuda'):
                out = model(X); loss = criterion(out, y)
            scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        else:
            out = model(X); loss = criterion(out, y)
            loss.backward(); optimizer.step()
        total_loss += loss.item() * X.size(0)
        preds.extend(torch.sigmoid(out).detach().cpu().numpy().flatten())
        targets.extend(y.cpu().numpy().flatten())
    return total_loss / len(loader.dataset), roc_auc_score(targets, preds)

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, preds, targets = 0.0, [], []
    with torch.no_grad():
        for X, y in loader:
            X = X.to(device, non_blocking=True)
            y = y.float().unsqueeze(1).to(device, non_blocking=True)
            with torch.amp.autocast('cuda') if device.type == 'cuda' else torch.no_grad():
                out = model(X); loss = criterion(out, y)
            total_loss += loss.item() * X.size(0)
            preds.extend(torch.sigmoid(out).cpu().numpy().flatten())
            targets.extend(y.cpu().numpy().flatten())
    p = np.array(preds); t = np.array(targets)
    return total_loss / len(loader.dataset), accuracy_score(t, p >= 0.5), roc_auc_score(t, p)

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler    = torch.amp.GradScaler('cuda') if DEVICE.type == 'cuda' else None
ckpt_path = f'{OUTPUT_DIR}/best_efficientnet_b0.pt'

best_val_auc = 0.0
history = []

print('=' * 72)
print(f'STARTING EFFICIENTNET-B0 TRAINING ({EPOCHS} Epochs | Batch {BATCH_SIZE} | lr {LR})')
print('=' * 72)

total_start = time.time()
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_auc = train_epoch(model, train_loader, criterion, optimizer, scaler, DEVICE)
    scheduler.step()
    vl_loss, vl_acc, vl_auc = evaluate(model, val_loader, criterion, DEVICE)
    dur = time.time() - t0
    history.append({'epoch': epoch, 'train_loss': tr_loss, 'train_auc': tr_auc, 'val_loss': vl_loss, 'val_acc': vl_acc, 'val_auc': vl_auc})
    star = '  ★ NEW BEST' if vl_auc > best_val_auc else ''
    print(f'  [{epoch:02d}/{EPOCHS}] ({dur:.0f}s)  TrLoss={tr_loss:.4f} TrAUC={tr_auc:.4f} || ValAcc={vl_acc*100:6.2f}%  ValAUC={vl_auc:.4f}{star}')
    if vl_auc > best_val_auc:
        best_val_auc = vl_auc
        raw_model = model.module if hasattr(model, 'module') else model
        torch.save(raw_model.state_dict(), ckpt_path)

print('=' * 72)
print(f'Training Completed in {(time.time() - total_start)/60:.1f} min | Peak Val AUC: {best_val_auc:.4f}')
print(f'Saved Best Checkpoint: {ckpt_path}')
print('=' * 72)

STARTING EFFICIENTNET-B0 TRAINING (15 Epochs | Batch 128 | lr 0.0003)
  [01/15] (28s)  TrLoss=0.5214 TrAUC=0.8142 || ValAcc= 79.12%  ValAUC=0.8715  ★ NEW BEST
  [02/15] (27s)  TrLoss=0.4421 TrAUC=0.8710 || ValAcc= 80.84%  ValAUC=0.8884  ★ NEW BEST
  [03/15] (27s)  TrLoss=0.3984 TrAUC=0.8975 || ValAcc= 81.60%  ValAUC=0.8972  ★ NEW BEST
  [04/15] (27s)  TrLoss=0.3645 TrAUC=0.9168 || ValAcc= 82.08%  ValAUC=0.9024  ★ NEW BEST
  [05/15] (27s)  TrLoss=0.3328 TrAUC=0.9324 || ValAcc= 82.52%  ValAUC=0.9068  ★ NEW BEST
  [06/15] (27s)  TrLoss=0.3012 TrAUC=0.9465 || ValAcc= 82.80%  ValAUC=0.9094  ★ NEW BEST
  [07/15] (27s)  TrLoss=0.2741 TrAUC=0.9578 || ValAcc= 83.12%  ValAUC=0.9120  ★ NEW BEST
  [08/15] (27s)  TrLoss=0.2486 TrAUC=0.9674 || ValAcc= 83.24%  ValAUC=0.9135  ★ NEW BEST
  [09/15] (27s)  TrLoss=0.2241 TrAUC=0.9752 || ValAcc= 83.36%  ValAUC=0.9142  ★ NEW BEST
  [10/15] (27s)  TrLoss=0.1984 TrAUC=0.9821 || ValAcc= 83.48%  ValAUC=0.9150  ★ NEW BEST
  [11/15] (27s)  TrLoss=0.1765 TrAUC=0.9

In [7]:
print(f'Loading best checkpoint for test evaluation: {ckpt_path}')
eval_model = build_efficientnet_b0().to(DEVICE)
eval_model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
eval_model.eval()

t_preds, t_targets = [], []
with torch.no_grad():
    for X, y in test_loader:
        X = X.to(DEVICE, non_blocking=True)
        out = eval_model(X)
        t_preds.extend(torch.sigmoid(out).cpu().numpy().flatten())
        t_targets.extend(y.numpy().flatten())

t_preds   = np.array(t_preds)
t_targets = np.array(t_targets).astype(int)
t_binary  = (t_preds >= 0.5).astype(int)

test_acc = accuracy_score(t_targets, t_binary)
test_auc = roc_auc_score(t_targets, t_preds)
pc, rc, _ = precision_recall_curve(t_targets, t_preds)
test_pr  = auc(rc, pc)
cm       = confusion_matrix(t_targets, t_binary)
tn, fp, fn, tp = cm.ravel()
sens  = tp / (tp + fn) if (tp + fn) > 0 else 0
spec  = tn / (tn + fp) if (tn + fp) > 0 else 0
prec  = tp / (tp + fp) if (tp + fp) > 0 else 0
f1    = 2 * prec * sens / (prec + sens) if (prec + sens) > 0 else 0
brier = brier_score_loss(t_targets, t_preds)

print('\n' + '=' * 68)
print('  FINAL SEALED TEST COHORT EVALUATION — EFFICIENTNET-B0')
print('=' * 68)
print(f'  Test Accuracy : {test_acc*100:.2f}%  (ResNet-18: 84.94% | ResNet-50: 82.14%)')
print(f'  ROC-AUC       : {test_auc:.4f}      (ResNet-18: 0.9392)')
print(f'  PR-AUC        : {test_pr:.4f}      (ResNet-18: 0.9455)')
print(f'  Sensitivity   : {sens*100:.2f}%')
print(f'  Specificity   : {spec*100:.2f}%')
print(f'  Precision     : {prec*100:.2f}%')
print(f'  F1-Score      : {f1:.4f}')
print(f'  Brier Score   : {brier:.4f}      (Lower is better calibration)')
print('-' * 68)
print(f'  Confusion Matrix: TN={tn:,} | FP={fp:,} | FN={fn:,} | TP={tp:,}')
print('=' * 68)

Loading best checkpoint for test evaluation: /kaggle/working/outputs/efficientnet/best_efficientnet_b0.pt

  FINAL SEALED TEST COHORT EVALUATION — EFFICIENTNET-B0
  Test Accuracy : 82.64%  (ResNet-18: 84.94% | ResNet-50: 82.14%)
  ROC-AUC       : 0.9086      (ResNet-18: 0.9392)
  PR-AUC        : 0.9015      (ResNet-18: 0.9455)
  Sensitivity   : 78.80%
  Specificity   : 86.48%
  Precision     : 85.35%
  F1-Score      : 0.8106
  Brier Score   : 0.1314      (Lower is better calibration)
--------------------------------------------------------------------
  Confusion Matrix: TN=2,162 | FP=338 | FN=530 | TP=1,970


In [8]:
grand_leaderboard = pd.DataFrame([
    {'Model': '★ EfficientNet-B0 (Central Focus Track B)', 'Test Acc': f'{test_acc*100:.2f}%', 'ROC-AUC': f'{test_auc:.4f}', 'F1-Score': f'{f1:.4f}', 'Brier Score': f'{brier:.4f}'},
    {'Model': 'ResNet-18 (Deep CNN Full96)',              'Test Acc': '84.94%',                    'ROC-AUC': '0.9392',           'F1-Score': '0.8329',           'Brier Score': '0.1168'},
    {'Model': 'ResNet-50 (Deep CNN Full96)',              'Test Acc': '82.14%',                    'ROC-AUC': '0.9285',           'F1-Score': '0.7910',           'Brier Score': '0.1378'},
    {'Model': 'Hybrid Blend 50/50 (Best Classical)',      'Test Acc': '79.76%',                    'ROC-AUC': '0.8853',           'F1-Score': '0.7831',           'Brier Score': '0.1426'},
    {'Model': 'Hybrid Blend 70/30 (RBF/HistGBM)',         'Test Acc': '79.56%',                    'ROC-AUC': '0.8804',           'F1-Score': '0.7796',           'Brier Score': '0.1453'},
    {'Model': 'HistGradientBoosting (Handcrafted 59D)',    'Test Acc': '78.82%',                    'ROC-AUC': '0.8845',           'F1-Score': '0.7723',           'Brier Score': '0.1460'},
    {'Model': 'Default RBF Kernel SVM (Handcrafted 59D)', 'Test Acc': '78.80%',                    'ROC-AUC': '0.8746',           'F1-Score': '0.7756',           'Brier Score': '0.1493'},
    {'Model': 'Tuned Random Forest (150 trees)',          'Test Acc': '78.46%',                    'ROC-AUC': '0.8863',           'F1-Score': '0.7615',           'Brier Score': '0.1467'},
    {'Model': 'Logistic Regression Baseline',             'Test Acc': '75.84%',                    'ROC-AUC': '0.8373',           'F1-Score': '0.7460',           'Brier Score': '0.1679'},
    {'Model': 'Linear SVM Baseline',                      'Test Acc': '75.46%',                    'ROC-AUC': '0.8292',           'F1-Score': '0.7435',           'Brier Score': '0.1850'},
])

print('\n' + '=' * 84)
print('          MASTER PCAM BENCHMARK LEADERBOARD (Identical Split & Seed=42)')
print('=' * 84)
print(grand_leaderboard.to_string(index=False))
print('=' * 84)


          MASTER PCAM BENCHMARK LEADERBOARD (Identical Split & Seed=42)
                                    Model Test Acc ROC-AUC F1-Score Brier Score
★ EfficientNet-B0 (Central Focus Track B)   82.64%  0.9086   0.8106      0.1314
              ResNet-18 (Deep CNN Full96)   84.94%  0.9392   0.8329      0.1168
              ResNet-50 (Deep CNN Full96)   82.14%  0.9285   0.7910      0.1378
      Hybrid Blend 50/50 (Best Classical)   79.76%  0.8853   0.7831      0.1426
         Hybrid Blend 70/30 (RBF/HistGBM)   79.56%  0.8804   0.7796      0.1453
   HistGradientBoosting (Handcrafted 59D)   78.82%  0.8845   0.7723      0.1460
 Default RBF Kernel SVM (Handcrafted 59D)   78.80%  0.8746   0.7756      0.1493
          Tuned Random Forest (150 trees)   78.46%  0.8863   0.7615      0.1467
             Logistic Regression Baseline   75.84%  0.8373   0.7460      0.1679
                      Linear SVM Baseline   75.46%  0.8292   0.7435      0.1850
